# V1 prototype (the pasted cell), verbatim, on this machine — 26-value grid

The prototype side the user specified: the V1 `PatternSearchCV` class and its
"#Execute Pattern Search:" cell exactly as recorded (default scoring = R^2,
`clf` with `n_jobs=-1`, no refit/scoring arguments) — the configuration that
produced Run A (18 evals, R^2 0.809981, 867.34 s on the original machine).

Disclosed mechanical shims (plumbing only; the search loop is byte-identical):
`sklearn.utils._joblib` -> `joblib`; base class = the notebook's own
sklearn-1.x-updated `BaseSearchCV2` (V1's original base predates sklearn 1.0);
`iid=` dropped from one super() call; `error_score='raise-deprecating'` ->
`'raise'`; `df.append` -> `pd.concat` (removed in pandas 2). Data path localized.

In [1]:
# --- Data pipeline: byte-for-byte replication of the prototype notebook ---
import time
import numpy as np
import pandas as pd

t0 = time.time()
trainBench = pd.read_csv(r"C:\FILES\Code\Benchmarking\train.csv")  # notebook: /home/dell/train.csv

SplitPoint = int(len(trainBench.index) * 0.8)
print("SplitPoint: ", SplitPoint)

df = trainBench

# int64 -> int32; object -> category -> numeric codes (Date included, as in the prototype)
Int64columns = df.select_dtypes(["int64"]).columns
df[Int64columns] = df[Int64columns].astype(np.int32)
cat_columns = df.select_dtypes(["object"]).columns
df[cat_columns] = df[cat_columns].astype("category")
df[cat_columns] = df[cat_columns].apply(lambda x: x.cat.codes)

# the five weather columns the prototype drops (note the dataset's own 'kM' typo)
df = df.drop("Max_Gust_SpeedKm_h", axis=1)
df = df.drop("CloudCover", axis=1)
df = df.drop("Max_VisibilityKm", axis=1)
df = df.drop("Min_VisibilitykM", axis=1)
df = df.drop("Mean_VisibilityKm", axis=1)

# positional 80/20 chronological split
trainBench, validBench = df.iloc[:SplitPoint, :], df.iloc[SplitPoint:, :]
print("Training Set shape", trainBench.shape)
print("Validation Set shape", validBench.shape)
del df

# feature mask: everything but the target (alphabetical order, incl. Date codes
# and NumberOfCustomers, exactly as the prototype had it)
mask = trainBench.columns.difference(["NumberOfSales"])
trainDataset_X = trainBench[mask]
trainDataset_y = trainBench["NumberOfSales"]
validBench_X = validBench[mask]
validBench_y = validBench["NumberOfSales"]
print("Feature Columns:")
print(list(mask))
del trainBench, validBench
print(f"Pipeline done in {time.time() - t0:.1f} s")

SplitPoint:  418416


C:\Users\Home\AppData\Local\Temp\ipykernel_16268\4255920482.py:17: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_columns = df.select_dtypes(["object"]).columns


Training Set shape (418416, 31)
Validation Set shape (104605, 31)
Feature Columns:
['AssortmentType', 'Date', 'Events', 'HasPromotions', 'IsHoliday', 'IsOpen', 'Max_Dew_PointC', 'Max_Humidity', 'Max_Sea_Level_PressurehPa', 'Max_TemperatureC', 'Max_Wind_SpeedKm_h', 'Mean_Dew_PointC', 'Mean_Humidity', 'Mean_Sea_Level_PressurehPa', 'Mean_TemperatureC', 'Mean_Wind_SpeedKm_h', 'Min_Dew_PointC', 'Min_Humidity', 'Min_Sea_Level_PressurehPa', 'Min_TemperatureC', 'NearestCompetitor', 'NumberOfCustomers', 'Precipitationmm', 'Region', 'Region_AreaKM2', 'Region_GDP', 'Region_PopulationK', 'StoreID', 'StoreType', 'WindDirDegrees']
Pipeline done in 4.3 s


In [2]:
import time
import warnings
import numpy as np
from numpy.ma import MaskedArray # Correct import for MaskedArray
import pandas as pd
from scipy.stats import rankdata

from sklearn.base import BaseEstimator, is_classifier, clone, MetaEstimatorMixin
from sklearn.model_selection import check_cv # Moved to model_selection
# Internal validation functions - check path stability
from sklearn.model_selection._validation import _fit_and_score, _aggregate_score_dicts
from sklearn.exceptions import NotFittedError
# Parallel and delayed are often exposed via sklearn.utils._joblib now
from joblib import Parallel, delayed  # SHIM: sklearn.utils._joblib removed in sklearn>=1.3
from sklearn.utils import check_random_state
# from sklearn.utils.fixes import MaskedArray # Removed
from sklearn.utils.validation import indexable, check_is_fitted
# from sklearn.utils.metaestimators import if_delegate_has_method # Removed
# Scorer-related functions might be directly in metrics or _scorer
from sklearn.metrics import check_scoring
from sklearn.metrics._scorer import _check_multimetric_scoring # Check this path

from abc import ABCMeta, abstractmethod
from collections import defaultdict
from collections.abc import Mapping, Sequence, Iterable
from functools import partial, reduce
from itertools import product
import operator

In [3]:
# --- Custom BaseSearchCV2 ---
# WARNING: Relying on internal scikit-learn APIs like _fit_and_score is fragile
# and may break without warning in future scikit-learn updates.
class BaseSearchCV2(BaseEstimator, MetaEstimatorMixin, metaclass=ABCMeta):
    """Abstract base class for hyper parameter search with cross-validation.
       Updated attempt for scikit-learn >= 1.0, removing iid and if_delegate_has_method.
       Includes fixes for score_params argument and dictionary output of _fit_and_score.
    """

    @abstractmethod
    def __init__(self, estimator, *, scoring=None, n_jobs=None, # iid removed
                 refit=True, cv='warn', verbose=0, pre_dispatch='2*n_jobs',
                 error_score='raise', # Changed default from 'raise-deprecating'
                 return_train_score=True):

        self.scoring = scoring
        self.estimator = estimator
        self.n_jobs = n_jobs
        # self.iid = iid # Removed iid
        self.refit = refit
        self.cv = cv
        self.verbose = verbose
        self.pre_dispatch = pre_dispatch
        self.error_score = error_score
        self.return_train_score = return_train_score

    @property
    def _estimator_type(self):
        return self.estimator._estimator_type

    def score(self, X, y=None):
        """Returns the score on the given data, if the estimator has been refit."""
        self._check_is_fitted('score')
        # Use self.scorer_ which is set during fit
        if not hasattr(self, 'scorer_') or self.scorer_ is None:
             # Fallback to estimator's score method if scorer_ isn't set (should be set in fit)
             if hasattr(self.best_estimator_, 'score'):
                 return self.best_estimator_.score(X, y)
             else:
                 raise ValueError("No score function explicitly defined, "
                                  "and the estimator doesn't provide one.")

        # Handle multi-metric vs single-metric scoring
        # self.scorer_ is the callable (single) or dict (multi)
        score_func = self.scorer_[self.refit] if self.multimetric_ else self.scorer_
        return score_func(self.best_estimator_, X, y)


    def _check_is_fitted(self, method_name):
        if not self.refit:
            raise NotFittedError(f'This {type(self).__name__} instance was initialized '
                                 'with refit=False. {method_name} is '
                                 'available only after refitting on the best '
                                 'parameters. You can refit an estimator '
                                 'manually using the ``best_params_`` attribute')
        else:
            # Ensure 'best_estimator_' attribute exists
            check_is_fitted(self, 'best_estimator_')

    # Methods delegating to best_estimator_
    # Decorators @if_delegate_has_method removed
    def predict(self, X):
        """Call predict on the estimator with the best found parameters."""
        self._check_is_fitted('predict')
        return self.best_estimator_.predict(X)

    def predict_proba(self, X):
        """Call predict_proba on the estimator with the best found parameters."""
        self._check_is_fitted('predict_proba')
        return self.best_estimator_.predict_proba(X)

    def predict_log_proba(self, X):
        """Call predict_log_proba on the estimator with the best found parameters."""
        self._check_is_fitted('predict_log_proba')
        return self.best_estimator_.predict_log_proba(X)

    def decision_function(self, X):
        """Call decision_function on the estimator with the best found parameters."""
        self._check_is_fitted('decision_function')
        return self.best_estimator_.decision_function(X)

    def transform(self, X):
        """Call transform on the estimator with the best found parameters."""
        self._check_is_fitted('transform')
        return self.best_estimator_.transform(X)

    def inverse_transform(self, Xt):
        """Call inverse_transform on the estimator with the best found params."""
        self._check_is_fitted('inverse_transform')
        return self.best_estimator_.inverse_transform(Xt)

    @property
    def classes_(self):
        self._check_is_fitted("classes_")
        return self.best_estimator_.classes_

    @abstractmethod
    def _run_search(self, evaluate_candidates):
        """Mandatory method to be implemented by subclasses."""
        raise NotImplementedError("_run_search not implemented.")

    def fit(self, X, y=None, *, groups=None, **fit_params):
        """Run fit with all sets of parameters."""
        estimator = self.estimator
        # Use default cv=5 if 'warn'
        cv = 5 if self.cv == 'warn' else self.cv
        cv = check_cv(cv, y, classifier=is_classifier(estimator))

        # --- SCORER SETUP (Final Revision) ---
        scoring_input = self.scoring
        if scoring_input is None:
             if hasattr(self.estimator, "score"):
                 _scorer_arg_for_fit_and_score = check_scoring(self.estimator, scoring=None)
                 scorer_name = "score" # Default name when using estimator's score
                 scorers_dict = {scorer_name: _scorer_arg_for_fit_and_score}
                 self.multimetric_ = False
                 self.scorer_ = _scorer_arg_for_fit_and_score
                 refit_metric = scorer_name
             else:
                 raise ValueError("If scoring is None, the estimator must have a score method.")
        elif isinstance(scoring_input, str) or callable(scoring_input):
             # Single metric case
             try:
                 single_scorer_callable = check_scoring(self.estimator, scoring=scoring_input)
                 _scorer_arg_for_fit_and_score = single_scorer_callable

                 # --- Get the actual scorer name ---
                 if isinstance(scoring_input, str):
                      scorer_name = scoring_input
                 else: # It's a callable, use a default name
                      scorer_name = "score"
                 # --- Use scorer_name as the key and for refit_metric ---
                 scorers_dict = {scorer_name: single_scorer_callable}
                 self.multimetric_ = False
                 self.scorer_ = single_scorer_callable
                 refit_metric = scorer_name # <-- Use actual name

             except Exception as e:
                 print(f"Error during single metric scoring setup ('{scoring_input}'): {e}")
                 raise e
        else:
             # Multi-metric case (list, tuple, dict)
             try:
                 scorers_dict, self.multimetric_ = _check_multimetric_scoring(
                     self.estimator, scoring=scoring_input
                 )
                 _scorer_arg_for_fit_and_score = scorers_dict
                 self.scorer_ = scorers_dict
                 # Check refit validity for multi-metric
                 if self.refit is not False and (
                         not isinstance(self.refit, str) or
                         self.refit not in scorers_dict) and not callable(self.refit):
                     raise ValueError("For multi-metric scoring, the parameter refit must be set "
                                      "to a scorer key or a callable. If refit is not needed, "
                                      "set refit=False. %r was passed." % self.refit)
                 refit_metric = self.refit # Use self.refit directly here
             except Exception as e:
                 print(f"Error during multi-metric scoring setup: {e}")
                 raise e
        # --- END SCORER SETUP ---


        X, y, groups = indexable(X, y, groups)
        n_splits = cv.get_n_splits(X, y, groups)

        base_estimator = clone(self.estimator)

        parallel = Parallel(n_jobs=self.n_jobs, verbose=self.verbose,
                            pre_dispatch=self.pre_dispatch)

        # Pass the determined scorer argument (callable or dict) to _fit_and_score
        fit_and_score_kwargs = dict(scorer=_scorer_arg_for_fit_and_score, # Correct arg type passed
                                    fit_params=fit_params,
                                    return_train_score=self.return_train_score,
                                    return_n_test_samples=True,
                                    return_times=True,
                                    return_parameters=False,
                                    error_score=self.error_score,
                                    verbose=self.verbose,
                                    score_params=None)

        results = {}
        all_candidate_params = []
        all_out = []

        # Use a context manager for Parallel
        with parallel:
            def evaluate_candidates(candidate_params):
                candidate_params = list(candidate_params)
                n_candidates = len(candidate_params)

                if self.verbose > 0:
                    print(f"Fitting {n_splits} folds for each of {n_candidates} candidates, "
                          f"totalling {n_candidates * n_splits} fits")

                try:
                    out = parallel(delayed(_fit_and_score)(clone(base_estimator),
                                                           X, y,
                                                           train=train, test=test,
                                                           parameters=parameters,
                                                           **fit_and_score_kwargs)
                                   for parameters, (train, test)
                                   in product(candidate_params,
                                              cv.split(X, y, groups)))
                except ImportError:
                     raise ImportError("Failed to import or use internal function "
                                       "'_fit_and_score'. This custom SearchCV implementation "
                                       "is likely incompatible with the current scikit-learn version.")
                except Exception as e:
                     print(f"Error during parallel execution of _fit_and_score: {e}")
                     raise e


                if not out:
                    raise ValueError('No fits were performed. Was the CV iterator empty? '
                                     'Were there no candidates?')
                elif len(out) != n_candidates * n_splits:
                     print(f"Warning: Expected {n_candidates * n_splits} results, but got {len(out)}. "
                           "This might be due to error_score='raise'.")

                all_candidate_params.extend(candidate_params)
                all_out.extend(out)

                nonlocal results
                # Pass the 'scorers_dict' (always a dict with correct keys) for formatting results
                results = self._format_results(
                    all_candidate_params, scorers_dict, n_splits, all_out)
                return results

            self._run_search(evaluate_candidates)

        # Post-search processing (ranking, best estimator)
        if not results:
             print("Warning: No results were obtained from the search.")
             return self

        # Determine best candidate based on 'refit_metric' determined earlier
        if self.refit or not self.multimetric_:
            if callable(self.refit):
                try:
                    self.best_index_ = self.refit(results)
                    if not isinstance(self.best_index_, int):
                        raise TypeError('best_index_ returned is not an integer')
                    if not (0 <= self.best_index_ < len(results["params"])):
                        raise IndexError('best_index_ index out of range')
                except Exception as e:
                    print(f"Error calling custom refit function: {e}")
                    raise e
            else:
                # Ensure the refit_metric exists in the results
                mean_test_score_key = f"mean_test_{refit_metric}" # Uses correct refit_metric now
                rank_test_score_key = f"rank_test_{refit_metric}" # Uses correct refit_metric now

                if rank_test_score_key not in results:
                     if not any(k.startswith("rank_test_") for k in results.keys()):
                          print("Warning: No rank keys found in cv_results_. Cannot determine best parameters.")
                          self.cv_results_ = results
                          self.n_splits_ = n_splits
                          return self
                     else:
                          available_rank_keys = [k for k in results.keys() if k.startswith("rank_test_")]
                          raise ValueError(f"Ranking key '{rank_test_score_key}' not found in cv_results_. "
                                           f"Ensure scoring includes '{refit_metric}' or refit is a valid scorer key. "
                                           f"Available rank keys: {available_rank_keys}")


                try:
                    # Find the index where rank is 1 (best)
                    if rank_test_score_key not in results or np.all(np.isnan(results[rank_test_score_key])):
                         print(f"Warning: Rank key '{rank_test_score_key}' missing or all NaN. Attempting fallback using nanargmax on mean score.")
                         self.best_index_ = np.nanargmax(results[mean_test_score_key])
                    else:
                         valid_ranks = results[rank_test_score_key][~np.isnan(results[rank_test_score_key])]
                         if 1 not in valid_ranks:
                              print(f"Warning: Rank 1 not found for metric '{refit_metric}'. Attempting fallback using nanargmax.")
                              self.best_index_ = np.nanargmax(results[mean_test_score_key])
                         else:
                              best_indices = np.where(results[rank_test_score_key] == 1)[0]
                              if len(best_indices) == 0:
                                   print(f"Warning: Rank 1 not found unexpectedly for metric '{refit_metric}'. Attempting fallback using nanargmax.")
                                   self.best_index_ = np.nanargmax(results[mean_test_score_key])
                              else:
                                   self.best_index_ = best_indices[0]

                    self.best_score_ = results[mean_test_score_key][self.best_index_]

                except (IndexError, ValueError) as e:
                     print(f"Warning: Could not find best index for metric '{refit_metric}' even with fallbacks. Error: {e}")
                     if mean_test_score_key in results:
                          try:
                              self.best_index_ = np.nanargmax(results[mean_test_score_key])
                              self.best_score_ = results[mean_test_score_key][self.best_index_]
                              print(f"Final fallback successful: Found best index {self.best_index_} using nanargmax.")
                          except (ValueError, IndexError) as fallback_e:
                              raise ValueError(f"Could not find best index for metric '{refit_metric}'. Check cv_results_. Final fallback error: {fallback_e}")
                     else:
                          raise ValueError(f"Cannot determine best index. Mean score key '{mean_test_score_key}' not found.")
                except KeyError:
                     raise KeyError(f"Metric '{refit_metric}' not found in cv_results_. Available metrics: "
                                    f"{[k for k in results if k.startswith('mean_test_')]}")

            if hasattr(self, 'best_index_') and self.best_index_ < len(results["params"]):
                 self.best_params_ = results["params"][self.best_index_]
            else:
                 print("Warning: Could not determine best parameters due to index issue.")
                 self.best_params_ = None


        # Refit the best estimator on the whole dataset
        if self.refit and hasattr(self, 'best_params_') and self.best_params_ is not None:
            self.best_estimator_ = clone(base_estimator).set_params(**self.best_params_)
            refit_start_time = time.time()
            if y is not None:
                self.best_estimator_.fit(X, y, **fit_params)
            else:
                self.best_estimator_.fit(X, **fit_params)
            refit_end_time = time.time()
            self.refit_time_ = refit_end_time - refit_start_time
        elif self.refit:
             print("Warning: Refit skipped because best parameters could not be determined.")


        # Store final results
        self.cv_results_ = results
        self.n_splits_ = n_splits

        return self

    def _format_results(self, candidate_params, scorers, n_splits, out):
        """Format results from _fit_and_score into cv_results_ dictionary.
           Updated for scikit-learn >= 1.0 where _fit_and_score returns a dict.
        """
        n_candidates = len(candidate_params)

        if not out:
            # print("Warning: _format_results received empty 'out' list.") # Keep warnings minimal
            return {}

        # --- DEBUG PRINTS REMOVED ---

        # --- UNPACKING FROM LIST OF DICTIONARIES ---
        train_score_dicts_list = []
        test_score_dicts_list = []
        test_sample_counts_list = []
        fit_time_list = []
        score_time_list = []
        processed_elements = 0

        expected_tuple_size = 5 if self.return_train_score else 4 # Still useful for context in warnings

        for i, result_dict in enumerate(out):
            if isinstance(result_dict, dict):
                processed_elements += 1
                try:
                    test_scores = result_dict.get('test_scores', {})
                    if not isinstance(test_scores, dict):
                         scorer_key = list(scorers.keys())[0] if len(scorers) == 1 else 'score'
                         test_scores = {scorer_key: test_scores}

                    if self.return_train_score:
                        train_scores = result_dict.get('train_scores', {})
                        if not isinstance(train_scores, dict):
                             scorer_key = list(scorers.keys())[0] if len(scorers) == 1 else 'score'
                             train_scores = {scorer_key: train_scores}
                        train_score_dicts_list.append(train_scores)

                    n_test_samples = result_dict.get('n_test_samples', np.nan)
                    fit_time_val = result_dict.get('fit_time', np.nan)
                    score_time_val = result_dict.get('score_time', np.nan)

                    test_score_dicts_list.append(test_scores)
                    test_sample_counts_list.append(n_test_samples)
                    fit_time_list.append(fit_time_val)
                    score_time_list.append(score_time_val)

                    fit_error = result_dict.get('fit_error', None)
                    if fit_error is not None:
                         print(f"Warning: Fit error reported for element {i}: {fit_error}")

                except Exception as e:
                     print(f"Warning: Error processing result dictionary element {i}: {e}. Value: {result_dict}. Skipping.")

            else:
                print(f"ERROR: Element {i} in 'out' is not a dictionary. "
                      f"Got Type: {type(result_dict)}, Value: {str(result_dict)[:500]}... Skipping.")
        # --- END UNPACKING ---

        expected_results = n_candidates * n_splits
        if processed_elements == 0:
             print("ERROR: No valid results could be unpacked from 'out'. Cannot format results.")
             return {}
        elif processed_elements < expected_results:
             print(f"Warning: Only {processed_elements}/{expected_results} results were successfully unpacked. "
                   "This might indicate errors during fitting/scoring for some folds/candidates.")

        # --- AGGREGATION AND STORAGE ---
        try:
            test_scores_agg = _aggregate_score_dicts(test_score_dicts_list)
            if self.return_train_score and train_score_dicts_list:
                train_scores_agg = _aggregate_score_dicts(train_score_dicts_list)
            else:
                train_scores_agg = None
        except Exception as e:
             print(f"Error during score aggregation: {e}")
             return {}

        results = {}

        # Helper function _store
        def _store(key_name, array_list, n_splits_effective, n_candidates_effective, weights=None, splits=False, rank=False):
            """Store metrics in the results dictionary. Adjusted for potentially fewer results."""
            orig_n_candidates = len(candidate_params)
            orig_n_splits = n_splits

            is_empty = False
            if array_list is None: is_empty = True
            elif isinstance(array_list, (list, np.ndarray)):
                current_size = array_list.size if isinstance(array_list, np.ndarray) else len(array_list)
                if current_size == 0: is_empty = True
            else:
                 # print(f"Warning: Unexpected type for array_list in _store for key '{key_name}': {type(array_list)}. Treating as empty.")
                 is_empty = True

            if is_empty:
                 # print(f"Warning: Empty list/array provided for key '{key_name}'. Filling results with NaN.")
                 results[f'mean_{key_name}'] = np.full(orig_n_candidates, np.nan)
                 results[f'std_{key_name}'] = np.full(orig_n_candidates, np.nan)
                 if rank: results[f"rank_{key_name}"] = np.full(orig_n_candidates, np.nan)
                 if splits:
                     for split_i in range(orig_n_splits): results[f"split{split_i}_{key_name}"] = np.full(orig_n_candidates, np.nan)
                 return

            try:
                if not isinstance(array_list, np.ndarray): array_list = np.array(array_list, dtype=np.float64)
                array = array_list.reshape(n_candidates_effective, n_splits_effective)
            except ValueError as e:
                # print(f"ERROR: Could not reshape array for key '{key_name}'. "
                #       f"Expected {n_candidates_effective * n_splits_effective} elements from {array_list.size} valid results. Error: {e}. Filling with NaN.")
                results[f'mean_{key_name}'] = np.full(orig_n_candidates, np.nan)
                results[f'std_{key_name}'] = np.full(orig_n_candidates, np.nan)
                if rank: results[f"rank_{key_name}"] = np.full(orig_n_candidates, np.nan)
                if splits:
                    for split_i in range(orig_n_splits): results[f"split{split_i}_{key_name}"] = np.full(orig_n_candidates, np.nan)
                return

            with warnings.catch_warnings():
                 warnings.simplefilter("ignore", category=RuntimeWarning)
                 array_means_effective = np.ma.average(np.ma.masked_invalid(array), axis=1, weights=weights).filled(np.nan)
                 array_stds_effective = np.nanstd(array, axis=1)

            array_means = np.full(orig_n_candidates, np.nan)
            array_stds = np.full(orig_n_candidates, np.nan)
            fill_len = min(orig_n_candidates, n_candidates_effective)
            array_means[:fill_len] = array_means_effective[:fill_len]
            array_stds[:fill_len] = array_stds_effective[:fill_len]

            results[f'mean_{key_name}'] = array_means
            results[f'std_{key_name}'] = array_stds

            if splits:
                 for split_i in range(orig_n_splits):
                     split_data = np.full(orig_n_candidates, np.nan)
                     if split_i < n_splits_effective:
                         if array.shape[1] > split_i: split_data[:fill_len] = array[:fill_len, split_i]
                     results[f"split{split_i}_{key_name}"] = split_data

            if rank:
                try:
                    # For any scikit-learn scorer, a higher value is better.
                    # To give rank 1 to the highest score, we rank the negated array of scores.
                    rank_scores = -array_means
                    results[f"rank_{key_name}"] = rankdata(rank_scores, method='min').astype(int)
                except ValueError:
                    print(f"Warning: Could not compute rank for key '{key_name}'.")
                    results[f"rank_{key_name}"] = np.full(orig_n_candidates, np.nan)

        # Determine effective dimensions
        if processed_elements == 0: n_candidates_effective, n_splits_effective = 0, 0
        elif processed_elements % n_splits == 0:
             n_candidates_effective = processed_elements // n_splits
             n_splits_effective = n_splits
        elif processed_elements < n_candidates * n_splits:
             n_candidates_effective = n_candidates
             n_splits_effective = n_splits
             # print(f"DEBUG: Assuming {n_candidates_effective} candidates and {n_splits_effective} splits for reshaping, based on {processed_elements} results.")
        else: n_candidates_effective, n_splits_effective = n_candidates, n_splits

        _store('fit_time', fit_time_list, n_splits_effective, n_candidates_effective)
        _store('score_time', score_time_list, n_splits_effective, n_candidates_effective)

        param_results = defaultdict(lambda: MaskedArray(np.empty(n_candidates,), mask=True, dtype=object))
        for cand_i, params in enumerate(candidate_params):
            for name, value in params.items(): param_results[f"param_{name}"][cand_i] = value
        results.update(param_results)
        results['params'] = candidate_params

        try:
            if len(test_sample_counts_list) == n_candidates_effective * n_splits_effective:
                 test_sample_counts_arr = np.array(test_sample_counts_list).reshape(n_candidates_effective, n_splits_effective)
                 weights = test_sample_counts_arr[0]
            else: weights = None
        except ValueError as e: weights = None

        for scorer_name in scorers.keys():
            if scorer_name in test_scores_agg:
                 _store(f'test_{scorer_name}', test_scores_agg[scorer_name], n_splits_effective, n_candidates_effective, splits=True, rank=True, weights=None)
            else:
                 # print(f"Warning: Test score for '{scorer_name}' not found in aggregated results.")
                 results[f'mean_test_{scorer_name}'] = np.full(n_candidates, np.nan)
                 results[f'std_test_{scorer_name}'] = np.full(n_candidates, np.nan)
                 results[f'rank_test_{scorer_name}'] = np.full(n_candidates, np.nan)
                 for split_i in range(n_splits): results[f"split{split_i}_test_{scorer_name}"] = np.full(n_candidates, np.nan)

            if self.return_train_score and train_scores_agg:
                 if scorer_name in train_scores_agg:
                     _store(f'train_{scorer_name}', train_scores_agg[scorer_name], n_splits_effective, n_candidates_effective, splits=True, weights=None)
                 else:
                     # print(f"Warning: Train score for '{scorer_name}' not found in aggregated results.")
                     results[f'mean_train_{scorer_name}'] = np.full(n_candidates, np.nan)
                     results[f'std_train_{scorer_name}'] = np.full(n_candidates, np.nan)
                     for split_i in range(n_splits): results[f"split{split_i}_train_{scorer_name}"] = np.full(n_candidates, np.nan)

        return results

In [4]:
#Run This cell and then the "#Execute Pattern Search:" cell
class PatternSearchCV(BaseSearchCV2):
    _required_parameters = ["estimator", "param_distributions"]

    def __init__(self, estimator, param_distributions, scoring=None, #n_iter=10,
                 n_jobs=None, iid='warn', refit=True,
                 cv='warn', verbose=0, pre_dispatch='2*n_jobs',
                 random_state=None, error_score='raise',
                 return_train_score=False):
        self.param_distributions = param_distributions
        #self.n_iter = n_iter
        self.random_state = random_state
        super().__init__(
            estimator=estimator, scoring=scoring,
            n_jobs=n_jobs, refit=refit, cv=cv, verbose=verbose,
            pre_dispatch=pre_dispatch, error_score=error_score,
            return_train_score=return_train_score)
        self.ResultDf = pd.DataFrame()

#     param_dist={'n_estimators':[230,240,250],
#            'max_features':[2,3],
#            'max_depth':[16,17,30]}

        class Dimension():
            def __init__(self, value):
                #If value is a Tuple, divide the interval into "length" equaly spaced intervals:
                if isinstance(value,tuple):
                   lower=value[0];upper=value[1];length=value[2]
                   value=[lower + x*(upper-lower)/(length-1) for x in range(length)]
                self.value=value
                self.value.sort()
                self.min=self.value[0]
                self.max=self.value[-1]
                self.midptidx=int((len(self.value)/2)-0.5)
                self.midpoint=self.value[self.midptidx]
                self.Delta=(len(value)-1)-self.midptidx
                self.BestValue=self.midpoint
                self.CurrIndex=self.midptidx
                self.BestIndex=self.midptidx

        self.Space={}
        for Dkey, Dval in param_dist.items():
            self.Space[Dkey]=Dimension(Dval)
            print(Dkey,":",Dval)

    def _run_search(self, evaluate_candidates):
        """Search best parameters using Pattern Search Method"""
        alfa=2;Episilon=0.001 ;k=0
        Ndimensions=len(self.Space);

        xc={}; xd={}; xb={}
        #builds the first exploratory point by collecting the midpoint of each dimension:
        for Dkey, Dval in self.Space.items():
            xc[Dkey]=Dval.midpoint

        for CurDim in range(0,Ndimensions): #divide Delta by 2
            #print("Dividing delta by 2:")
            list(self.Space.values())[CurDim].Delta=int(0.5+list(self.Space.values())[CurDim].Delta/alfa)

        DeltaVector=[]
        for CurDim in range(0,Ndimensions):
            DeltaVector.append(list(self.Space.values())[CurDim].Delta)
        print(" ")
        print("Current Delta values for each dimension: ", DeltaVector)

        #Evaluate the first point as the midpoint of the search space:
        BestScore = evaluate_candidates([xc])['mean_test_score'][-1]
        print(" ")
        print(xc)

        cols=list(xc.keys())
        cols.append('score')
        df=pd.DataFrame(columns=cols)
        xd=xc.copy()
        xd.update({'score':BestScore})
        df=pd.concat([df, pd.DataFrame([xd])], ignore_index=True)
        print(df)

        BestIdx=0; InitialExploration=0


        # i=exploratory moves iterations; k=Overall Iterations
        i=0; Continue=0

        while Continue<3:
            #Exploratory Search:
            while i < Ndimensions:
                k+=1
                #print("Iteration k:",k)
                print("Dimension: ",i+1)
                for Direction in [1,-1]:
                    xn={}; xd={}
                    for CurDim in range(0,Ndimensions): #Build the vextor xn:
                        if i == CurDim:
                            NewIndex=list(self.Space.values())[CurDim].CurrIndex + Direction*list(self.Space.values())[CurDim].Delta
                            if NewIndex>len(list(self.Space.values())[CurDim].value)-1: NewIndex=len(list(self.Space.values())[CurDim].value)-1
                            if NewIndex<0: NewIndex=0
                            xn[list(self.Space.keys())[CurDim]]=list(self.Space.values())[CurDim].value[NewIndex]
                        else:
                            CurrInd=list(self.Space.values())[CurDim].CurrIndex
                            if CurrInd>len(list(self.Space.values())[CurDim].value)-1: CurrInd=len(list(self.Space.values())[CurDim].value)-1
                            if CurrInd<0: CurrInd =0
                            xn[list(self.Space.keys())[CurDim]]=list(self.Space.values())[CurDim].value[CurrInd]
                    if list(xn.values()) not in df.drop(['score'], axis=1).values.tolist():
                        print(Direction, xn)
                        CurrScore = evaluate_candidates([xn])['mean_test_score'][-1]
                        #cols=list(xn.keys())
                        #cols.append('score')
                        xd=xn.copy()
                        xd.update({'score':CurrScore})
                        df=pd.concat([df, pd.DataFrame([xd])], ignore_index=True)
                        print(df)
                        #print(CurrScore)
                        if CurrScore > BestScore:
                            BestScore=CurrScore
                            list(self.Space.values())[i].BestValue=list(self.Space.values())[i].value[NewIndex]
                            list(self.Space.values())[i].BestIndex=NewIndex
                            xb=xn.copy()
                            BestIdx=k-1
                            #break
                #list(self.Space.values())[i].CurrIndex=list(self.Space.values())[i].BestIndex
                i+=1
            xd={}
            NotEqual=False
            for Dkey, Dval in self.Space.items():
                #print(Dval.CurrIndex,Dval.BestIndex)
                if Dval.CurrIndex != Dval.BestIndex: NotEqual=True
            print(" ");print(" ");
            #print("NotEqual =",NotEqual)
            DeltaVector=[]
            if NotEqual:
                for CurDim in range(0,Ndimensions): #update Current index to Best Point:
                    list(self.Space.values())[CurDim].CurrIndex=list(self.Space.values())[CurDim].BestIndex
                    #xd[list(self.Space.keys())[CurDim]]=list(self.Space.values())[CurDim].BestValue
                    #BestIndex=list(self.Space.values())[CurDim].BestIndex
                    #xn[list(self.Space.keys())[CurDim]]=list(self.Space.values())[CurDim].value[BestIndex]
                #print("xn: ",xn)
                #print("xd:", xd)
                print("Best vector so far (xb):",xb)
                #print(df.loc[df['score'].idxmax()])
                #Multiply delta by 2
                print("multiplying delta by 2:")
                for CurDim in range(0,Ndimensions):
                    list(self.Space.values())[CurDim].Delta=int(0.5+list(self.Space.values())[CurDim].Delta*alfa)
                    DeltaVector.append(list(self.Space.values())[CurDim].Delta)
            else:
                print("Dividing delta by 2:")
                for CurDim in range(0,Ndimensions): #Stay in current point and divide Delta by 2
                    list(self.Space.values())[CurDim].Delta=int(0.5+list(self.Space.values())[CurDim].Delta/alfa)
                    DeltaVector.append(list(self.Space.values())[CurDim].Delta)

            print("Current Delta values for each dimension: ",DeltaVector)
            i=0
            if DeltaVector==list(np.ones(3)): Continue+=1

        print(" ");print(" ")
        print("Best Parameters Found:")
        print(df.loc[df['score'].idxmax()])
        print(" ");print(" ")
        self.ResultDf=df


In [5]:
import time as time_module
t_v1_start = time_module.time()

In [6]:
#Execute Pattern Search:
import numpy as np
import time  # SHIM 6
from scipy.stats import randint as sp_randint

from sklearn.model_selection import RandomizedSearchCV
#from sklearn.ensemble import RandomForestRegressor
from sklearn.ensemble import ExtraTreesRegressor

# Load data
SplitPercent=100
Nrows,_=trainDataset_X.shape
SplitPoint=int(Nrows*(SplitPercent/100))
X, y = trainDataset_X.iloc[:SplitPoint, :], trainDataset_y.iloc[:SplitPoint]


# build model

clf = ExtraTreesRegressor(n_estimators=240,
                                 max_features = int(X.columns.size - 15),
                                 max_depth = 16,
                                 n_jobs=-1,
                                 random_state=0)

# specify parameters and distributions to sample from

param_dist={#'n_estimators':[20,25,30,35],
           'max_features':[2,3,4],
           'n_estimators': list(np.linspace(0, 250, num=26, endpoint=True,dtype=int)+10),
           #'lr':list(np.logspace(-2, -5, num=4, endpoint=True, base=10.0)),
           'max_depth':list(range(5,18,1))}

# Utility function to report best scores
def report(results, n_top=3):
    for i in range(1, n_top + 1):
        candidates = np.flatnonzero(results['rank_test_score'] == i)
        for candidate in candidates:
            print("Model with rank: {0}".format(i))
            print("Mean validation score: {0:.3f} (std: {1:.3f})".format(
                  results['mean_test_score'][candidate],
                  results['std_test_score'][candidate]))
            print("Parameters: {0}".format(results['params'][candidate]))
            print("")

from sklearn.model_selection import TimeSeriesSplit
print("Using Time Series Cross Validation")
tscv = TimeSeriesSplit(n_splits=5)
#scores = cross_validate(forest, trainDataset_X, trainDataset_y, cv=tscv, scoring=('r2','explained_variance','neg_mean_absolute_error','neg_mean_squared_error') )


# run randomized search
pattern_search = PatternSearchCV(clf, param_distributions=param_dist,
                                    cv=tscv, n_jobs=-1, verbose=0)

start = time.time()
pattern_search.fit(X, y)
print("PatternSearchCV took %.2f seconds " % ((time.time() - start)))
report(pattern_search.cv_results_)

Using Time Series Cross Validation
max_features : [2, 3, 4]
n_estimators : [np.int64(10), np.int64(20), np.int64(30), np.int64(40), np.int64(50), np.int64(60), np.int64(70), np.int64(80), np.int64(90), np.int64(100), np.int64(110), np.int64(120), np.int64(130), np.int64(140), np.int64(150), np.int64(160), np.int64(170), np.int64(180), np.int64(190), np.int64(200), np.int64(210), np.int64(220), np.int64(230), np.int64(240), np.int64(250), np.int64(260)]
max_depth : [5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17]
 
Current Delta values for each dimension:  [1, 7, 3]


 
{'max_features': 3, 'n_estimators': np.int64(130), 'max_depth': 11}
  max_features n_estimators max_depth     score
0            3          130        11  0.671751
Dimension:  1
1 {'max_features': 4, 'n_estimators': np.int64(130), 'max_depth': 11}


  max_features n_estimators max_depth     score
0            3          130        11  0.671751
1            4          130        11  0.721118
-1 {'max_features': 2, 'n_estimators': np.int64(130), 'max_depth': 11}


  max_features n_estimators max_depth     score
0            3          130        11  0.671751
1            4          130        11  0.721118
2            2          130        11  0.611482
Dimension:  2
1 {'max_features': 3, 'n_estimators': np.int64(200), 'max_depth': 11}


  max_features n_estimators max_depth     score
0            3          130        11  0.671751
1            4          130        11  0.721118
2            2          130        11  0.611482
3            3          200        11  0.670294
-1 {'max_features': 3, 'n_estimators': np.int64(60), 'max_depth': 11}


...

   max_features n_estimators max_depth     score
0             3          130        11  0.671751
1             4          130        11  0.721118
2             2          130        11  0.611482
3             3          200        11  0.670294
4             3           60        11  0.662625
5             3          130        14  0.722283
6             3          130         8  0.596943
7             4          130        14  0.774317
8             2          130        14  0.669796
9             4          260        14  0.764782
10            4           10        14  0.736537
11            4          130        17  0.809692
12            4          130         8   0.66277
13            2          130        17  0.721477
14            4          260        17   0.80324
15            4           10        17  0.778177
16            4          130         5  0.575108
17            3          130        17  0.766341
18            4          200        17  0.805551
19            4     

   max_features n_estimators max_depth     score
0             3          130        11  0.671751
1             4          130        11  0.721118
2             2          130        11  0.611482
3             3          200        11  0.670294
4             3           60        11  0.662625
5             3          130        14  0.722283
6             3          130         8  0.596943
7             4          130        14  0.774317
8             2          130        14  0.669796
9             4          260        14  0.764782
10            4           10        14  0.736537
11            4          130        17  0.809692
12            4          130         8   0.66277
13            2          130        17  0.721477
14            4          260        17   0.80324
15            4           10        17  0.778177
16            4          130         5  0.575108
17            3          130        17  0.766341
18            4          200        17  0.805551
19            4     

   max_features n_estimators max_depth     score
0             3          130        11  0.671751
1             4          130        11  0.721118
2             2          130        11  0.611482
3             3          200        11  0.670294
4             3           60        11  0.662625
5             3          130        14  0.722283
6             3          130         8  0.596943
7             4          130        14  0.774317
8             2          130        14  0.669796
9             4          260        14  0.764782
10            4           10        14  0.736537
11            4          130        17  0.809692
12            4          130         8   0.66277
13            2          130        17  0.721477
14            4          260        17   0.80324
15            4           10        17  0.778177
16            4          130         5  0.575108
17            3          130        17  0.766341
18            4          200        17  0.805551
19            4     

   max_features n_estimators max_depth     score
0             3          130        11  0.671751
1             4          130        11  0.721118
2             2          130        11  0.611482
3             3          200        11  0.670294
4             3           60        11  0.662625
5             3          130        14  0.722283
6             3          130         8  0.596943
7             4          130        14  0.774317
8             2          130        14  0.669796
9             4          260        14  0.764782
10            4           10        14  0.736537
11            4          130        17  0.809692
12            4          130         8   0.66277
13            2          130        17  0.721477
14            4          260        17   0.80324
15            4           10        17  0.778177
16            4          130         5  0.575108
17            3          130        17  0.766341
18            4          200        17  0.805551
19            4     

PatternSearchCV took 1710.57 seconds 
Model with rank: 1
Mean validation score: 0.810 (std: 0.018)
Parameters: {'max_features': 4, 'n_estimators': np.int64(150), 'max_depth': 17}

Model with rank: 2
Mean validation score: 0.810 (std: 0.018)
Parameters: {'max_features': 4, 'n_estimators': np.int64(130), 'max_depth': 17}

Model with rank: 3
Mean validation score: 0.809 (std: 0.018)
Parameters: {'max_features': 4, 'n_estimators': np.int64(140), 'max_depth': 17}



In [7]:
# metrics: V1 prototype (bugs intact, default R^2 scoring, as pasted)
from sklearn.model_selection import cross_validate
v1_elapsed = time_module.time() - t_v1_start
v1_n_fits = len(pattern_search.ResultDf)
v1_best = {k: int(v) if float(v).is_integer() else v
           for k, v in pattern_search.best_params_.items()}
v1_r2 = pattern_search.best_score_

clf_best = ExtraTreesRegressor(n_jobs=-1, random_state=0, **v1_best)
cvres = cross_validate(clf_best, X, y, cv=tscv,
                       scoring=("r2", "neg_mean_absolute_error"), n_jobs=-1)
v1_mae = -cvres["test_neg_mean_absolute_error"].mean()
v1_r2_check = cvres["test_r2"].mean()
print("=" * 68)
print("V1 PROTOTYPE (as pasted; R^2 objective) - 26-value grid, this machine")
print("=" * 68)
print(f"evaluations          : {v1_n_fits}")
print(f"full-fit equivalents : {float(v1_n_fits):.2f}")
print(f"wall-clock           : {v1_elapsed:.2f} s")
print(f"best point           : {v1_best}")
print(f"best CV R2 (search)  : {v1_r2:.6f}   (re-check {v1_r2_check:.6f})")
print(f"best CV MAE          : {v1_mae:.3f}")
print()
print("Run A reference (original machine): 18 evals, R2 0.809981 at (4,150,17), 867.34 s")
print("full trajectory (order of evaluation):")
print(pattern_search.ResultDf.to_string())

V1 PROTOTYPE (as pasted; R^2 objective) - 26-value grid, this machine
evaluations          : 33
full-fit equivalents : 33.00
wall-clock           : 1710.90 s
best point           : {'max_features': 4, 'n_estimators': 150, 'max_depth': 17}
best CV R2 (search)  : 0.809981   (re-check 0.809981)
best CV MAE          : 805.730

Run A reference (original machine): 18 evals, R2 0.809981 at (4,150,17), 867.34 s
full trajectory (order of evaluation):
   max_features n_estimators max_depth     score
0             3          130        11  0.671751
1             4          130        11  0.721118
2             2          130        11  0.611482
3             3          200        11  0.670294
4             3           60        11  0.662625
5             3          130        14  0.722283
6             3          130         8  0.596943
7             4          130        14  0.774317
8             2          130        14  0.669796
9             4          260        14  0.764782
10            4